In [22]:
import pandas as pd
from PIL.TiffImagePlugin import X_RESOLUTION

nltk.download('wordnet')
true_df = pd.read_csv("True.csv")
fake_df = pd.read_csv("Fake.csv")

true_df["label"] = 1
fake_df["label"] = 0

df = pd.concat([true_df, fake_df], axis=0)

# combine title and text
df["content"] = df["title"] + " " + df["text"]

print(df.head())

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...


                                               title  \
0  As U.S. budget fight looms, Republicans flip t...   
1  U.S. military to accept transgender recruits o...   
2  Senior U.S. Republican senator: 'Let Mr. Muell...   
3  FBI Russia probe helped by Australian diplomat...   
4  Trump wants Postal Service to charge 'much mor...   

                                                text       subject  \
0  WASHINGTON (Reuters) - The head of a conservat...  politicsNews   
1  WASHINGTON (Reuters) - Transgender people will...  politicsNews   
2  WASHINGTON (Reuters) - The special counsel inv...  politicsNews   
3  WASHINGTON (Reuters) - Trump campaign adviser ...  politicsNews   
4  SEATTLE/WASHINGTON (Reuters) - President Donal...  politicsNews   

                 date  label  \
0  December 31, 2017       1   
1  December 29, 2017       1   
2  December 31, 2017       1   
3  December 30, 2017       1   
4  December 29, 2017       1   

                                             cont

In [26]:
df = df[["content", "label"]]

In [27]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)        # remove URLs
    text = re.sub(r"[^a-zA-Z ]", "", text)     # remove punctuation & numbers
    text = re.sub(r"\s+", " ", text)           # remove extra spaces
    return text

df["content"] = df["content"].apply(clean_text)

C:\Users\User\AppData\Local\Temp\ipykernel_24008\3199607069.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["content"] = df["content"].apply(clean_text)


In [28]:
from nltk.tokenize import word_tokenize

df["tokens"] = df["content"].apply(word_tokenize)

C:\Users\User\AppData\Local\Temp\ipykernel_24008\3333909871.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["tokens"] = df["content"].apply(word_tokenize)


In [29]:
import re
def clean_text(text):
    text=text.lower()
    text=re.sub(r'https\s+',' ',text)
    text=re.sub('[^a-zA-Z]',' ',text)
    text=re.sub(r'\s+'," ",text)
    text=text.split()
    return text
df["content"]=df["content"].apply(clean_text)

C:\Users\User\AppData\Local\Temp\ipykernel_24008\2269255755.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["content"]=df["content"].apply(clean_text)


In [30]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

df["tokens"] = df["tokens"].apply(
    lambda words: [w for w in words if w not in stop_words]
)

C:\Users\User\AppData\Local\Temp\ipykernel_24008\2141246068.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["tokens"] = df["tokens"].apply(


In [31]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

df["tokens"] = df["tokens"].apply(
    lambda words: [lemmatizer.lemmatize(w) for w in words]
)

C:\Users\User\AppData\Local\Temp\ipykernel_24008\1221824835.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["tokens"] = df["tokens"].apply(


In [35]:
df["cleaned_tokens"] = df["tokens"].apply(lambda x:" ".join(x))

C:\Users\User\AppData\Local\Temp\ipykernel_24008\2912847558.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["cleaned_tokens"] = df["tokens"].apply(lambda x:" ".join(x))


In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer=TfidfVectorizer(max_features=500)

In [37]:
X=vectorizer.fit_transform(df["cleaned_tokens"])
y=df["label"]

In [38]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report

In [42]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=10)

In [43]:
model=LogisticRegression()
model.fit(X_train,y_train)
y_pred=model.predict(X_test)


In [44]:
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))
print(accuracy_score(y_test,y_pred))

[[4636   78]
 [  62 4204]]
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      4714
           1       0.98      0.99      0.98      4266

    accuracy                           0.98      8980
   macro avg       0.98      0.98      0.98      8980
weighted avg       0.98      0.98      0.98      8980

0.9844097995545658
